# 10. HyPE - Hypothetical Prompt Embeddings

Move the work to index time: embed questions each chunk could answer, not the chunk itself.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

HyDE moves work to *query time* (one LLM call per search). **HyPE** moves it to *index time*: for each chunk, the LLM writes the questions that chunk answers, and those question vectors are what you index. At query time the user's question is matched question-to-question, which is a much easier match, and no LLM call is needed.

## How it works

1. For every chunk, generate N hypothetical questions
2. Embed the questions (not the chunk text)
3. Remember which chunk each question came from
4. At query time embed the user question and find nearest hypothetical questions
5. Return the parent chunks (deduplicated)

```text
 INDEX TIME:  chunk --> LLM --> Q1, Q2, Q3 --> embed --> store (each points to chunk)
 QUERY TIME:  user question --> embed --> nearest hypothetical questions --> parent chunks --> LLM
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

HyDE pays an LLM call per query. **HyPE** generates hypothetical *questions* per chunk once, so query-vs-question matching is question-to-question and queries stay cheap.

In [ ]:
def hypothetical_questions(chunk, n=3):
    data = llm_json(f"Write {n} distinct questions that this passage answers. Reply with a JSON list of strings.\n\nPassage: {chunk}")
    return [x for x in (data or []) if isinstance(x, str)]

print(hypothetical_questions(chunks[0]))

In [ ]:
qs, owner = [], []
for i, chunk in enumerate(chunks):
    for q in hypothetical_questions(chunk):
        qs.append(q)
        owner.append({"chunk": i})
hype_store = build_store(qs, owner)
print(len(qs), "hypothetical questions indexed")

In [ ]:
def hype_search(query, k=3):
    seen, results = set(), []
    for h in hype_store.search(query, k * 3):
        c = h["meta"]["chunk"]
        if c not in seen:
            seen.add(c)
            results.append({"idx": c, "text": chunks[c], "score": h["score"], "meta": {"matched_question": h["text"]}})
    return results[:k]

query = "What does a cross-encoder do?"
for h in hype_search(query):
    print(f"{h['score']:.3f} matched: {h['meta']['matched_question']}")
print(answer(query, [h["text"] for h in hype_search(query)]))

## What did the LLM generate?
Inspect the questions for a few chunks. Good questions are specific and varied.

In [ ]:
for i in [0, 3, 8]:
    print(f"CHUNK {i}: {chunks[i][:100]}...")
    for q in hypothetical_questions(chunks[i]): print("   ?", q)

## Evaluate against plain chunk retrieval
The golden set has no LLM call at query time for HyPE, so it also shows the latency benefit.

In [ ]:
import time
t = time.time(); r_base = evaluate_retriever(store.search, k=3); t_base = time.time() - t
t = time.time(); r_hype = evaluate_retriever(hype_search, k=3); t_hype = time.time() - t
compare_retrievers({"chunk embeddings": store.search, "HyPE (question embeddings)": hype_search}, k=3)

## Index-size trade-off

In [ ]:
print(f"chunks: {len(chunks)}   hypothetical questions: {len(qs)}   ({len(qs) / len(chunks):.1f} per chunk)")

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| No query-time LLM call - same latency as plain RAG | Indexing needs one LLM call per chunk |
| Question-to-question matching is easy for embedders | Question quality depends on the LLM |
| Multiple questions per chunk give multiple entry points | Index grows with N questions per chunk |

## When to use it

Stable corpora that are queried often: pay the cost once at indexing and get cheap, accurate queries forever.

## Experiments to try

- Vary N (1, 3, 5) and plot hit rate vs index size.
- Store both chunk and question vectors (this is notebook 16).
- Ask the LLM for questions at different difficulty levels.

## Key takeaways

- Match questions to questions.
- Pay at indexing time, not query time.
- Always map back to the source chunk.

---
**Next:** [11. Contextual Chunk Headers](../11_contextual_chunk_headers/contextual_chunk_headers.ipynb)